In [ ]:
import csv
from pathlib import Path

import numpy as np
import pandas as pd

RAW_MATCH_FILE = Path('2026orwil - raw match.csv')
PICKLIST_FILE = Path('2026orwil - Picklist.csv')


def parse_float(value):
    if value is None:
        return 0.0
    text = str(value).strip()
    if not text:
        return 0.0
    try:
        return float(text)
    except ValueError:
        return 0.0


def team_number(team_key):
    if not team_key:
        return None
    digits = ''.join(ch for ch in str(team_key) if ch.isdigit())
    if not digits:
        return None
    try:
        return int(digits)
    except ValueError:
        return None


def load_bps():
    if not PICKLIST_FILE.exists():
        raise SystemExit(f'missing picklist: {PICKLIST_FILE}')
    mapping = {}
    with PICKLIST_FILE.open(newline='', encoding='utf-8') as fh:
        reader = csv.DictReader(fh)
        for row in reader:
            team = team_number(row.get('key'))
            if team is None:
                continue
            mapping[team] = parse_float(row.get('avg_BPS'))
    return mapping


def compute_accuracy(predicted, actual):
    actual_abs = actual.abs()
    mask = actual_abs > 0
    accuracy = pd.Series(np.nan, index=predicted.index)
    accuracy.loc[mask] = 1.0 - (predicted[mask] - actual[mask]).abs() / actual_abs[mask]
    accuracy.loc[mask] = accuracy.loc[mask].clip(lower=0.0, upper=1.0)
    zero_mask = ~mask
    accuracy.loc[zero_mask] = (predicted[zero_mask].abs() == 0).astype(float)
    return accuracy


for required in (RAW_MATCH_FILE, PICKLIST_FILE):
    if not required.exists():
        raise SystemExit(f'missing required file: {required}')

raw_matches = pd.read_csv(RAW_MATCH_FILE)
raw_matches.columns = raw_matches.columns.str.strip()

raw_matches['team_key'] = (
    raw_matches['team_key']
    .fillna('UNKNOWN')
    .astype(str)
    .str.strip()
    .str.upper()
)
raw_matches.loc[raw_matches['team_key'] == '', 'team_key'] = 'UNKNOWN'

bps_map = load_bps()
raw_matches['team_number'] = raw_matches['team_key'].apply(team_number)
raw_matches['avg_bps'] = raw_matches['team_number'].map(bps_map).fillna(0.0)

numeric_fields = ['a_scoringTime', 't_scoringTime', 'a_Climb', 'end_Climb']
for field in numeric_fields:
    if field in raw_matches.columns:
        raw_matches[field] = raw_matches[field].apply(parse_float)
    else:
        raw_matches[field] = 0.0

raw_matches['a_fuel'] = raw_matches['a_scoringTime'] * raw_matches['avg_bps']
raw_matches['tele_fuel'] = raw_matches['t_scoringTime'] * raw_matches['avg_bps']
raw_matches['auto_points'] = raw_matches['a_fuel'] + raw_matches['a_Climb']
raw_matches['teleop_points'] = raw_matches['tele_fuel']
raw_matches['tower_points'] = raw_matches['a_Climb'] + raw_matches['end_Climb']

raw_matches['ba_totalAutoPoints'] = raw_matches['ba_totalAutoPoints'].apply(parse_float)
raw_matches['ba_totalTeleopPoints'] = raw_matches['ba_totalTeleopPoints'].apply(parse_float)
raw_matches['ba_totalTowerPoints'] = raw_matches['ba_totalTowerPoints'].apply(parse_float)

raw_matches['auto_error'] = (raw_matches['auto_points'] - raw_matches['ba_totalAutoPoints']).abs()
raw_matches['teleop_error'] = (raw_matches['teleop_points'] - raw_matches['ba_totalTeleopPoints']).abs()
raw_matches['tower_error'] = (raw_matches['tower_points'] - raw_matches['ba_totalTowerPoints']).abs()

raw_matches['auto_accuracy'] = compute_accuracy(raw_matches['auto_points'], raw_matches['ba_totalAutoPoints'])
raw_matches['teleop_accuracy'] = compute_accuracy(raw_matches['teleop_points'], raw_matches['ba_totalTeleopPoints'])
raw_matches['tower_accuracy'] = compute_accuracy(raw_matches['tower_points'], raw_matches['ba_totalTowerPoints'])

summary = (
    raw_matches
    .groupby('team_key', dropna=False)
    .agg(
        auto_accuracy=('auto_accuracy', 'mean'),
        teleop_accuracy=('teleop_accuracy', 'mean'),
        tower_accuracy=('tower_accuracy', 'mean'),
        auto_error=('auto_error', 'mean'),
        teleop_error=('teleop_error', 'mean'),
        tower_error=('tower_error', 'mean'),
        observations=('key', 'count'),
        matches=('key', 'nunique'),
    )
    .reset_index()
    .sort_values('auto_accuracy', ascending=False)
)

print('per-scout accuracy summary:')
print(summary[['team_key', 'auto_accuracy', 'teleop_accuracy', 'tower_accuracy', 'observations']])

overall = raw_matches[['auto_accuracy', 'teleop_accuracy', 'tower_accuracy']].mean()
print('overall accuracy (auto / teleop / tower):')
print(overall)  



per-scout accuracy summary:
    team_key  auto_accuracy  teleop_accuracy  tower_accuracy  observations
1    FRC1425       0.739817         0.582047        0.555556            18
2    FRC1540       0.573095         0.565584        0.588235            17
24   FRC6696       0.536925         0.418715        0.588235            17
4    FRC2374       0.535061         0.557909        0.785714            14
20   FRC5937       0.524090         0.664419        0.555556            18
3    FRC1595       0.486796         0.424806        0.857143            14
29    FRC957       0.479688         0.559524        0.769231            14
6    FRC2550       0.470038         0.106467        0.857143            14
23   FRC6343       0.388789         0.324086        0.733333            15
17   FRC4488       0.378227         0.467118        0.562500            16
18   FRC4513       0.372433         0.482945        0.846154            14
5    FRC2471       0.369036         0.422279        0.411765            

In [4]:
OUTPUT_FILE = Path('bps_accuracy_by_category.csv')
summary.to_csv(OUTPUT_FILE, index=False)
print(f'wrote {OUTPUT_FILE}')
summary.head()



wrote bps_accuracy_by_category.csv


,team_key,auto_accuracy,teleop_accuracy,tower_accuracy,auto_error,teleop_error,tower_error,observations,matches
1,FRC1425,0.739817,0.582047,0.555556,13.693062,58.296524,0.909091,18,18
2,FRC1540,0.573095,0.565584,0.588235,22.735336,88.419376,0.909091,17,17
24,FRC6696,0.536925,0.418715,0.588235,13.284509,73.072058,0.909091,17,17
4,FRC2374,0.535061,0.557909,0.785714,10.349466,52.709419,0.000000,14,14
20,FRC5937,0.524090,0.664419,0.555556,16.121605,45.801607,0.909091,18,18
